# PCR — Encoder-Decoder MHA + Standard LoRA + Random Negatives 



In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import json
import math
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPUs:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))


PyTorch: 2.11.0+cu130
CUDA: True
GPUs: 2
0 Tesla T4
1 Tesla T4


In [2]:
# -------------------------
# Configuration
# -------------------------
SEED = 42

MAX_SEQ_LEN = 1024
CHUNK_OVERLAP = 64
CHUNK_CONTENT_LEN = MAX_SEQ_LEN - 2
CHUNK_STRIDE = CHUNK_CONTENT_LEN - CHUNK_OVERLAP
MAX_CHUNKS = 8

CHUNK_BATCH_SIZE = 8

NUM_NEGATIVES = 4

BATCH_SIZE = 8

TEMPERATURE = 0.07
EPOCHS = 5
GRAD_CLIP = 1.0
WEIGHT_DECAY = 0.01

# Standard additive LoRA
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Equal learning rates = standard LoRA.
LORA_A_LR = 2e-5
LORA_B_LR = 2e-5

# Memory-saving settings
USE_GRAD_CHECKPOINTING = True
USE_SDPA = True
USE_AMP = True

EVAL_DOCUMENT_BATCH_SIZE = 1

TOKENIZER_PATH = "/kaggle/input/datasets/rogue2222/tokenizer-results/project_bpe_16k.pkl"



HF_REPO = "Exploration-Lab/CS-781-Capstone"

TRAIN_JSON = "Phase_7_PCR/Train/train.json"
TRAIN_CANDIDATES_JSON = "Phase_7_PCR/Train/train_candidates.json"
DEV_JSON = "Phase_7_PCR/Dev/dev.json"
DEV_CANDIDATES_JSON = "Phase_7_PCR/Dev/dev_candidates.json"
TEST_JSON = "Phase_7_PCR/Test/test.json"
TEST_CANDIDATES_JSON = "Phase_7_PCR/Test/test_candidates.json"

CHECKPOINT_DIR = "/kaggle/working/PCR_EncoderDecoder_LoRA_Random_Negatives"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

RANDOM_NEGATIVE_CACHE_PATH = os.path.join(
    CHECKPOINT_DIR, "random_train_negative_map.json"
)
HISTORY_PATH = os.path.join(CHECKPOINT_DIR, "history.json")
BEST_CHECKPOINT = os.path.join(CHECKPOINT_DIR, "best.pt")
TEST_PREDICTIONS_PATH = os.path.join(
    CHECKPOINT_DIR, "test_retrieval_predictions.json"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("CHUNK_BATCH_SIZE:", CHUNK_BATCH_SIZE)
print("BATCH_SIZE:", BATCH_SIZE)
print("USE_GRAD_CHECKPOINTING:", USE_GRAD_CHECKPOINTING)
print("USE_SDPA:", USE_SDPA)


device: cuda
CHUNK_BATCH_SIZE: 8
BATCH_SIZE: 8
USE_GRAD_CHECKPOINTING: True
USE_SDPA: True


## OOM-safe encoder-decoder PCR design

- **Encoder sequence:** 1024 tokens per chunk.
- **Decoder sequence:** 1 token (`BOS`) per chunk.
- The decoder's cross-attention still sees the complete encoder sequence.
- `CHUNK_BATCH_SIZE=1` is retained.
- Gradient checkpointing is retained for encoder and decoder blocks.
- Query/candidate documents are processed sequentially inside each training batch, so `BATCH_SIZE=2` does not put two full document graphs on the GPU simultaneously.


In [3]:
# -------------------------
# Utilities
# -------------------------
def clear_cuda_memory():
    gc.collect()
    if torch.cuda.is_available():
        for gpu_id in range(torch.cuda.device_count()):
            with torch.cuda.device(gpu_id):
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
                torch.cuda.reset_peak_memory_stats()

def gpu_memory_report(prefix=""):
    if not torch.cuda.is_available():
        print(prefix + "CUDA unavailable")
        return
    for i in range(torch.cuda.device_count()):
        with torch.cuda.device(i):
            allocated = torch.cuda.memory_allocated() / 1024**3
            reserved = torch.cuda.memory_reserved() / 1024**3
            free, total = torch.cuda.mem_get_info()
            print(
                f"{prefix}GPU {i}: allocated={allocated:.2f} GB, "
                f"reserved={reserved:.2f} GB, "
                f"free={free/1024**3:.2f} GB / total={total/1024**3:.2f} GB"
            )

clear_cuda_memory()
gpu_memory_report("Initial | ")


Initial | GPU 0: allocated=0.00 GB, reserved=0.00 GB, free=14.46 GB / total=14.56 GB
Initial | GPU 1: allocated=0.00 GB, reserved=0.00 GB, free=14.46 GB / total=14.56 GB


In [4]:
# ------------------------------------------------------------
# Hugging Face login
# ------------------------------------------------------------

from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Hugging Face login successful.")
# ------------------------------------------------------------
# Download Phase-7 PCR files
# ------------------------------------------------------------

TRAIN_FILES = [
    "Phase_7_PCR/Train/train.json",
    "Phase_7_PCR/Train/train_candidates.json",
]
DEV_FILES = [
    "Phase_7_PCR/Dev/dev.json",
    "Phase_7_PCR/Dev/dev_candidates.json",
]
TEST_FILES = [
    "Phase_7_PCR/Test/test.json",
    "Phase_7_PCR/Test/test_candidates.json",
]

def download_files(files):
    paths = {}
    for filename in files:
        paths[filename] = hf_hub_download(
            repo_id=HF_DATASET_REPO,
            filename=filename,
            repo_type="dataset",
            token=HF_TOKEN,
        )
        print(filename, "->", paths[filename])
    return paths

train_file_paths = download_files(TRAIN_FILES)
dev_file_paths = download_files(DEV_FILES)
test_file_paths = download_files(TEST_FILES)
# ------------------------------------------------------------
# Load raw JSON
# ------------------------------------------------------------

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_data_raw = load_json(train_file_paths["Phase_7_PCR/Train/train.json"])
train_candidates_raw = load_json(train_file_paths["Phase_7_PCR/Train/train_candidates.json"])

dev_data_raw = load_json(dev_file_paths["Phase_7_PCR/Dev/dev.json"])
dev_candidates_raw = load_json(dev_file_paths["Phase_7_PCR/Dev/dev_candidates.json"])

test_data_raw = load_json(test_file_paths["Phase_7_PCR/Test/test.json"])
test_candidates_raw = load_json(test_file_paths["Phase_7_PCR/Test/test_candidates.json"])

print("Train queries:", len(train_data_raw))
print("Train candidates:", len(train_candidates_raw))
print("Dev queries:", len(dev_data_raw))
print("Dev candidates:", len(dev_candidates_raw))
print("Test queries:", len(test_data_raw))
print("Test candidates:", len(test_candidates_raw))


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Train/train.json -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_7_PCR/Train/train.json


Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

... (8 outputs left out)

Phase_7_PCR/Test/test.json -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_7_PCR/Test/test.json


Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Test/test_candidates.json -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_7_PCR/Test/test_candidates.json
Train queries: 827
Train candidates: 4320
Dev queries: 118
Dev candidates: 1023
Test queries: 237
Test candidates: 1727


In [5]:
class Node:
    def __init__(self,value, previous, next, index):
        self.value=value
        self.previous=previous
        self.next=next
        self.index=index


class LinkedArray:
    def __init__(self, items):
        self._array=[None for _ in range(len(items))]
        previous=None
        for i in range(len(items)):
            node=Node(items[i], previous, None, i)
            if previous is not None:
                node.previous=previous
                previous.next=node
            self._array[i]=node
            previous=node

    def get_previous_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].previous == None:
            return None
        return self._array[index].previous.index

    def get_next_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].next == None:
            return None
        return self._array[index].next.index

    def get_by_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index] == None:
            return None
        return self._array[index].value

    def get_second_next_index(self, index):
        if self._array[index] == None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].next == None or self._array[index].next.next == None:
            return None
        return self._array[index].next.next.index

    def len(self):
        return len(self._array)
    def exists(self, index):
        return (
            0 <= index < len(self._array)
            and self._array[index] is not None
        )

    def replace_pair(self, index, new_item):
        if index > len(self._array)-2 or self._array[index] == None or self._array[index].next == None:
            raise ValueError("Invalid index ",index,new_item)
        self._array[index].value = new_item
        self._array[self._array[index].next.index] = None
        self._array[index].next = self._array[index].next.next
        if self._array[index].next is not None:
            self._array[index].next.previous = self._array[index]


class MaxPriorityMap:
    def __init__(self, heap_key, map_key):
        self._heap_key = heap_key
        self._map_key = map_key
        self._heap = []
        # Map of keys to indexes in the heap
        self._map = {}

    def get_max(self):
        if len(self._heap) == 0:
            raise ValueError("the heap is empty")
        return self._heap[0]

    def push(self, item):
        self._heap.append(item)
        map_key = self._map_key(item)
        if map_key in self._map:
            raise ValueError("Map key already exist")
        self._map[self._map_key(item)] = len(self._heap) - 1
        self._heapify_up(len(self._heap) - 1)

    def pop(self):
        if len(self._heap) == 0:
            raise ValueError("the heap is empty")
        self._swap(0, len(self._heap) - 1)
        item = self._heap.pop()
        map_key = self._map_key(item)
        self._map.pop(map_key)
        self._heapify_down(0)
        return item

    def contains(self, map_key):
        return map_key in self._map

    def len(self):
        if len(self._heap) != len(self._map):
            raise ValueError('Invalid state of the priority map')
        return len(self._heap)

    def delete_by_map_key(self, map_key):
        if map_key not in self._map:
            raise ValueError("Map key does not exist")
        heap_index = self._map[map_key]
        item = self._heap[heap_index]
        self._swap(heap_index, len(self._heap) - 1)
        self._heap.pop()
        self._map.pop(map_key)
        if (heap_index < len(self._heap)):
            self._heapify_down(heap_index)
            self._heapify_up(heap_index)
        return item

    def _heapify_up(self, i):
        while i > 0 and self._heap_key(self._heap[i]) > self._heap_key(self._heap[self._parent(i)]):
            self._swap(i, self._parent(i))
            i = self._parent(i)

    def _heapify_down(self, i):
        largest = i
        left = self._left_child(i)
        right = self._right_child(i)

        if left < len(self._heap) and self._heap_key(self._heap[left]) > self._heap_key(self._heap[largest]):
            largest = left

        if right < len(self._heap) and self._heap_key(self._heap[right]) > self._heap_key(self._heap[largest]):
            largest = right

        if largest != i:
            self._swap(i, largest)
            self._heapify_down(largest)

    def _parent(self, i):
        return (i - 1) // 2

    def _left_child(self, i):
        return 2 * i + 1

    def _right_child(self, i):
        return 2 * i + 2

    def _swap(self, i, j):
        self._map[self._map_key(self._heap[i])] = j
        self._map[self._map_key(self._heap[j])] = i
        self._heap[i], self._heap[j] = self._heap[j], self._heap[i]


class StatsEntry:
    def __init__(self, pair, positions):
        self.pair = pair
        self.positions = positions

class TokenizerTrainer:
        def __init__(self, tokens_list, itos, num_merges):
            self.tokens_list = tokens_list
            self.itos = itos
            self._num_merges = num_merges
            self.stoi={}

        def train(self, next_token):
            self._positions=[LinkedArray(tokens) for tokens in self.tokens_list]
            self.get_stats()
            merges_done=0
            print("requested merges",self._num_merges)
            while  merges_done<self._num_merges:
                if self._stats.len() == 0:
                   print("STOPPED: stats became empty")
                   print("Merges completed:", merges_done)
                   break
                merge_stat = self._stats.pop()
                (current_token , next_token) = self.update_itos_and_stoi(next_token, merge_stat)
                #iterate through posistions replace with merge pair
                for position in list(merge_stat.positions):

                    if position not in merge_stat.positions:
                        continue
                    input_index = position[0]
                    token_index = position[1]
                    # print("current_token",current_token)
                    if not self._positions[input_index].exists(token_index):
                        merge_stat.positions.discard(position)
                        continue
                    self.update_right_token(input_index, token_index, merge_stat, current_token)
                    self.update_left_token(input_index, token_index, merge_stat, current_token)
                    self._positions[input_index].replace_pair(token_index, current_token)
                    merge_stat.positions.remove((input_index, token_index))
                merges_done += 1
            return self.itos


        def update_right_token(self, input_index, token_index, merge_stat, current_token):
           position = self._positions[input_index]
           next_token_idx = position.get_next_index(token_index)
           right_token_idx = position.get_second_next_index(token_index)
           if right_token_idx == None:
               return
           pair = (merge_stat.pair[1], position.get_by_index(right_token_idx))
           self.remove_pair_from_stats(merge_stat, pair, input_index, next_token_idx)
           new_pair = (current_token, pair[1])
           self.add_position_to_pair(new_pair, input_index, token_index)

        def update_left_token(self, input_index, token_index, merge_stat, current_token):
           position = self._positions[input_index]
           left_token_idx = position.get_previous_index(token_index)
           if left_token_idx == None:
               return
           pair = ( position.get_by_index(left_token_idx), merge_stat.pair[0])
           self.remove_pair_from_stats(merge_stat, pair, input_index, left_token_idx)
           new_pair = (pair[0] , current_token)
           self.add_position_to_pair(new_pair, input_index, left_token_idx)

        def add_position_to_pair(self, new_pair, input_index, token_index):
            if self._stats.contains(new_pair):
                stat = self._stats.delete_by_map_key(new_pair)
            else:
                stat = StatsEntry(new_pair, set())

            stat.positions.add((input_index, token_index))
            self._stats.push(stat)


        def remove_pair_from_stats(self, merge_stat, pair, input_index, token_index):
           if pair == merge_stat.pair:
               merge_stat.positions.discard((input_index, token_index))
           else:
                if self._stats.contains(pair):
                    stat = self._stats.delete_by_map_key(pair)
                    stat.positions.discard((input_index, token_index))
                    if len(stat.positions) != 0:
                        self._stats.push(stat)


        def update_itos_and_stoi(self, next_token, merge_stat):
           merge_pair = self.itos[merge_stat.pair[0]] + self.itos[merge_stat.pair[1]]
           if merge_pair in self.stoi:
               current_token = self.stoi[merge_pair]
           else:
               current_token = next_token
               self.itos[current_token] = merge_pair
               self.stoi[merge_pair] = current_token
               next_token = current_token+1
           return  (current_token , next_token)

        def get_stats(self):
             self._stats = MaxPriorityMap(
                                heap_key=lambda item: (len(item.positions),tuple(-x for x in item.pair)),
                                map_key=lambda item: item.pair
                            )
             stats={}
             for  text_id in range(len(self.tokens_list)):
                 tokens = self.tokens_list[text_id]
                 for idx in range(len(tokens)-1):
                     pair = (tokens[idx], tokens[idx+1])
                     if pair not in stats:
                         stats[pair]=StatsEntry(pair, set())
                     stats[pair].positions.add((text_id, idx))
             for key in stats.keys():
                self._stats.push(stats[key])

class TokenNode:
    def __init__(self, byte_value, token, parent, children):
        self.parent = parent
        self.byte_value = byte_value
        self.token = token
        self.children = children

class ToTokensConverter:

    def __init__(self, itos):
        self.itos = itos
        self.unk_token=1
        self._init_token_tree()

    def to_tokens(self, strings):
        return [
            self._to_tokens(string)
            for string in strings
        ]

    def _to_tokens(self, string):

        words = string.split()
        tokens = []
        for word in words:
            data = word.encode("utf-8") + b"</w>"
            word_tokens = self._tokenize_word(data)
            tokens.extend(word_tokens)
        return tokens

    def _tokenize_word(self, data):

        tokens = []

        byte_index = 0

        while byte_index < len(data):

            current_nodes = self._token_roots

            last_node_with_token = None
            last_index_with_token = byte_index

            while byte_index < len(data):

                byte_value = data[byte_index]

                if byte_value in current_nodes:

                    node = current_nodes[byte_value]

                    if node.token is not None:
                        last_node_with_token = node
                        last_index_with_token = byte_index

                    current_nodes = node.children
                    byte_index += 1

                else:
                    break

            if last_node_with_token is None:
                tokens.append(self.unk_token)

                # Move forward by one byte
                byte_index += 1

            else:
                tokens.append(last_node_with_token.token)
                byte_index = last_index_with_token + 1

        return tokens

    def _init_token_tree(self):

        self._token_roots = {}

        for token, token_bytes in self.itos.items():

            if token < 4:
                continue

            current_nodes = self._token_roots
            last_node = None

            for byte_value in token_bytes:

                if byte_value in current_nodes:

                    last_node = current_nodes[byte_value]

                else:

                    last_node = TokenNode(
                        byte_value,
                        None,
                        last_node,
                        {}
                    )

                    current_nodes[byte_value] = last_node

                current_nodes = last_node.children

            if last_node.token is not None:
                raise KeyError("The token is already taken")

            last_node.token = token

class BPETokenizer:
    def __init__(self, num_merges):
        self._num_merges = num_merges
        self.itos = {}
    def get_vocab_size(self):
        return len(self.itos)
    def train(self, texts):
        self.create_base_vocab()

        tokens_list = self.get_ids_from_text(texts)
        # for tokens in tokens_list:
        #     print(tokens)
        trainer = TokenizerTrainer(
                        tokens_list,
                        self.itos,
                        self._num_merges
                    )

        trainer.train(261)

        return self.itos, tokens_list

    def encode(self, text):
        converter = ToTokensConverter(self.itos)
        return converter.to_tokens([text])[0]

    def decode(self, tokens):
        return (
                b"".join(
                    self.itos[token]
                    for token in tokens
                )
                .decode("utf-8", errors="replace")
                .replace("</w>", " ")
                .strip()
            )

    def create_base_vocab(self):
        self.itos = {
            0: b"<pad>",
            1: b"<unk>",
            2: b"<s>",
            3: b"</s>",
            260 : b"</w>"
        }
        for i in range(256):
            self.itos[i + 4] = bytes([i])

    def get_ids_from_text(self, texts):
        tokens_list=[]
        for text in texts:
            words = text.split()
            for word in words:
                word_bytes = [byte + 4 for byte in word.encode("utf-8")]
                word_bytes.append(260)
                tokens_list.append(word_bytes)
        return tokens_list


In [6]:
# ------------------------------------------------------------
# Load the existing BPE tokenizer
# ------------------------------------------------------------
import pickle
BPE_PATH = "/kaggle/input/datasets/ksaiveerendraprasad/tokenizer-results/project_bpe_16k.pkl"

with open(BPE_PATH, "rb") as f:
    tokenizer = pickle.load(f)

print("BPE vocab size:", len(tokenizer.itos))

converter = ToTokensConverter(tokenizer.itos)

PAD_ID = 0
UNK_ID = 1
BOS_ID = 2
EOS_ID = 3
VOCAB_SIZE = 8000

print("Special IDs:", PAD_ID, UNK_ID, BOS_ID, EOS_ID)


BPE vocab size: 7999
Special IDs: 0 1 2 3


In [7]:

def convert_text_to_ids(text):
    token_ids = converter.to_tokens([text])[0]
    if len(token_ids) == 0:
        return [UNK_ID]
    token_ids = [int(x) for x in token_ids]
    bad_ids = [x for x in token_ids if x < 0 or x >= VOCAB_SIZE]
    if bad_ids:
        raise ValueError(f"Invalid token ids: {bad_ids[:10]}")
    return token_ids

print("Vocabulary size:", VOCAB_SIZE)


Vocabulary size: 8000


In [8]:
# ------------------------------------------------------------
# Data processing — same style as the reference notebook
# No normalization is performed.
# ------------------------------------------------------------

def case_text_to_string(text):
    if isinstance(text, str):
        return text

    if isinstance(text, list):
        parts = []
        for item in text:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict) and "text" in item:
                parts.append(str(item["text"]))
            else:
                raise TypeError(
                    f"Unsupported text item: {type(item)}"
                )
        return " ".join(parts)

    raise TypeError(f"Unsupported text type: {type(text)}")


def prepare_records(raw_records, labeled=False):
    records = {}

    if isinstance(raw_records, dict):
        iterator = raw_records.items()
    elif isinstance(raw_records, list):
        iterator = (
            (
                row.get(
                    "id",
                    row.get("query_id", row.get("case_id")),
                ),
                row,
            )
            for row in raw_records
        )
    else:
        raise TypeError("Dataset must be a dict or list.")

    for raw_id, row in iterator:
        if raw_id is None:
            raise ValueError("A record has no case ID.")

        case_id = str(raw_id)

        if case_id in records:
            raise ValueError(f"Duplicate case ID: {case_id}")

        text = case_text_to_string(row["text"])

        if not text.strip():
            raise ValueError(f"Empty text for case {case_id}")

        item = {"text": text}

        if labeled:
            positives = row["relevant_candidates"]
            if not isinstance(positives, list):
                raise ValueError(
                    "relevant_candidates must be a list."
                )
            item["relevant_candidates"] = [
                str(cid) for cid in positives
            ]

        records[case_id] = item

    return records


# IMPORTANT:
# train_data/dev_data are QUERIES.
# train_candidates/dev_candidates are CANDIDATES.

train_data = prepare_records(
    train_data_raw,
    labeled=True,
)
train_candidates = prepare_records(
    train_candidates_raw,
)

dev_data = prepare_records(
    dev_data_raw,
    labeled=True,
)
dev_candidates = prepare_records(
    dev_candidates_raw,
)

test_data = prepare_records(test_data_raw)
test_candidates = prepare_records(test_candidates_raw)

print("Train:", len(train_data), "queries,", len(train_candidates), "candidates")
print("Dev:  ", len(dev_data), "queries,", len(dev_candidates), "candidates")
print("Test: ", len(test_data), "queries,", len(test_candidates), "candidates")


Train: 827 queries, 4320 candidates
Dev:   118 queries, 1023 candidates
Test:  237 queries, 1727 candidates


In [9]:
# -------------------------
# Chunking
# -------------------------
def chunk_token_ids(token_ids):
    content = list(token_ids)

    if not content:
        content = [UNK_ID]

    starts = list(range(
        0,
        max(len(content) - CHUNK_CONTENT_LEN, 0) + 1,
        CHUNK_STRIDE,
    ))

    last_start = max(len(content) - CHUNK_CONTENT_LEN, 0)
    if not starts or starts[-1] != last_start:
        starts.append(last_start)

    starts = sorted(set(starts))

    if len(starts) > MAX_CHUNKS:
        # Preserve first and last chunks while selecting intermediate chunks
        # approximately uniformly.
        idx = np.linspace(0, len(starts) - 1, MAX_CHUNKS).round().astype(int)
        starts = [starts[i] for i in idx]
        starts = sorted(set(starts))

    chunks = []
    for start in starts:
        piece = content[start:start + CHUNK_CONTENT_LEN]
        chunk = [BOS_ID] + piece + [EOS_ID]
        chunk = chunk[:MAX_SEQ_LEN]

        if len(chunk) < MAX_SEQ_LEN:
            chunk = chunk + [PAD_ID] * (MAX_SEQ_LEN - len(chunk))

        chunks.append(chunk)

    while len(chunks) < MAX_CHUNKS:
        chunks.append([PAD_ID] * MAX_SEQ_LEN)

    return torch.tensor(chunks[:MAX_CHUNKS], dtype=torch.long)

def encode_record_text(text):
    ids = convert_text_to_ids(text)
    return chunk_token_ids(ids)

# Precompute chunks. No text normalization is performed.
def build_chunk_records(records, desc):
    result = {}
    for doc_id, row in tqdm(records.items(), desc=desc):
        result[doc_id] = {
            "chunks": encode_record_text(row["text"])
        }
    return result

train_query_records = build_chunk_records(train_data, "Chunking train queries")
train_candidate_records = build_chunk_records(
    train_candidates, "Chunking train candidates"
)
dev_query_records = build_chunk_records(dev_data, "Chunking dev queries")
dev_candidate_records = build_chunk_records(
    dev_candidates, "Chunking dev candidates"
)
test_query_records = build_chunk_records(test_data, "Chunking test queries")
test_candidate_records = build_chunk_records(
    test_candidates, "Chunking test candidates"
)

print("Example query chunks:", next(iter(train_query_records.values()))["chunks"].shape)


Chunking train queries:   0%|          | 0/827 [00:00<?, ?it/s]

Chunking train candidates:   0%|          | 0/4320 [00:00<?, ?it/s]

Chunking dev queries:   0%|          | 0/118 [00:00<?, ?it/s]

Chunking dev candidates:   0%|          | 0/1023 [00:00<?, ?it/s]

Chunking test queries:   0%|          | 0/237 [00:00<?, ?it/s]

Chunking test candidates:   0%|          | 0/1727 [00:00<?, ?it/s]

Example query chunks: torch.Size([8, 1024])


In [10]:
# -------------------------
# Random negatives
# -------------------------
def build_random_negative_map(query_data, candidate_data, seed=SEED):
    rng = random.Random(seed)
    query_ids = list(query_data.keys())
    candidate_ids = list(candidate_data.keys())

    negative_map = {}

    for qid in tqdm(query_ids, desc="Sampling random negatives"):
        positives = list(dict.fromkeys(
            query_data[qid].get("relevant_candidates", [])
        ))
        positive_set = set(positives)

        eligible = [
            cid for cid in candidate_ids
            if cid not in positive_set
        ]

        if len(eligible) < NUM_NEGATIVES:
            raise RuntimeError(
                f"Query {qid} has only {len(eligible)} eligible negatives."
            )

        negative_map[qid] = rng.sample(eligible, NUM_NEGATIVES)

    return negative_map

if os.path.exists(RANDOM_NEGATIVE_CACHE_PATH):
    with open(RANDOM_NEGATIVE_CACHE_PATH, "r", encoding="utf-8") as f:
        random_negative_map = json.load(f)
    print("Loaded cached random negatives.")
else:
    random_negative_map = build_random_negative_map(
        train_data,
        train_candidates,
        seed=SEED,
    )
    with open(RANDOM_NEGATIVE_CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(random_negative_map, f, indent=2)
    print("Saved random negatives.")

print("Example:", next(iter(random_negative_map.items())))


Sampling random negatives:   0%|          | 0/827 [00:00<?, ?it/s]

Saved random negatives.
Example: ('0000000599', ['0000432540', '0000088171', '0001069919', '0000952560'])


In [11]:
# -------------------------
# PCR Dataset
# -------------------------
class PCRDataset(Dataset):
    def __init__(
        self,
        query_data,
        query_records,
        candidate_records,
        negative_map,
        num_negatives=4,
    ):
        self.query_data = query_data
        self.query_records = query_records
        self.candidate_records = candidate_records
        self.negative_map = negative_map
        self.num_negatives = num_negatives

        self.query_ids = [
            qid for qid, row in query_data.items()
            if row.get("relevant_candidates")
        ]

    def __len__(self):
        return len(self.query_ids)

    def __getitem__(self, idx):
        qid = self.query_ids[idx]
        row = self.query_data[qid]

        positives = list(dict.fromkeys(row["relevant_candidates"]))
        negatives = self.negative_map[qid][:self.num_negatives]

        candidate_ids = positives + negatives
        positive_mask = [True] * len(positives) + [False] * len(negatives)

        query_chunks = self.query_records[qid]["chunks"]

        candidate_chunks = torch.stack([
            self.candidate_records[cid]["chunks"]
            for cid in candidate_ids
        ])

        return {
            "qid": qid,
            "query_chunks": query_chunks,
            "candidate_chunks": candidate_chunks,
            "candidate_ids": candidate_ids,
            "positive_mask": torch.tensor(
                positive_mask, dtype=torch.bool
            ),
        }

train_dataset = PCRDataset(
    train_data,
    train_query_records,
    train_candidate_records,
    random_negative_map,
    NUM_NEGATIVES,
)

def collate_fn(batch):

    return batch

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)

print("Training queries:", len(train_dataset))


Training queries: 817


In [12]:
# ============================================================
# USER-SUPPLIED ENCODER-DECODER MHA ARCHITECTURE
# ============================================================
# This cell intentionally follows the supplied architecture:
#   RoPE
#   MultiHeadAttention
#   CrossAttention
#   EncoderLM
#   DecoderLM
#   EncoderDecoderTransformer
#
# PCR does NOT replace this with a decoder-only model.
# ============================================================

"""Rotary Positional Embeddings, implemented from scratch."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


def build_rope_cache(max_seq_len: int, d_head: int, base: float = 10000.0):
    assert d_head % 2 == 0, "RoPE requires an even head dimension"
    inverse_frequencies = torch.tensor(
        [1 / base ** (2 * i / d_head) for i in range(d_head // 2)],
        dtype=torch.float32,
    )
    positions = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(positions, inverse_frequencies)
    angles = torch.cat([freqs, freqs], dim=-1)
    return angles.cos(), angles.sin()


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    d_head = x.shape[-1]
    x1 = x[..., :d_head // 2]
    x2 = x[..., d_head // 2:]
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(
    x: torch.Tensor,
    cos: torch.Tensor,
    sin: torch.Tensor,
) -> torch.Tensor:
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    return x * cos + rotate_half(x) * sin


class MultiHeadAttention(nn.Module):
    """
    Multi-head attention implemented with PyTorch SDPA.

    Padding-mask semantics are unchanged from the supplied architecture:
        True  = valid token
        False = padding

    GQA is handled by SDPA with enable_gqa=True when needed.
    """
    def __init__(
        self,
        d_model,
        n_heads,
        n_kv_heads,
        max_seq_len,
        dropout=0.0,
        use_rope=False,
        attention_scale=None,
        is_causal=False,
    ):
        super().__init__()

        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0

        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        self.is_causal = is_causal
        self.attention_scale = (
            1.0 / math.sqrt(self.d_head)
            if attention_scale is None
            else float(attention_scale)
        )

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(
        self,
        x,
        padding_mask=None,
        past_key_value=None,
        use_cache=False,
        position_offset=0,
    ):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            cos = self.rope_cos[position_offset:position_offset + T]
            sin = self.rope_sin[position_offset:position_offset + T]
            q = apply_rope(q, cos, sin)
            k = apply_rope(k, cos, sin)

        if past_key_value is not None:
            past_k, past_v = past_key_value
            k = torch.cat((past_k, k), dim=2)
            v = torch.cat((past_v, v), dim=2)

        present = (k, v) if use_cache else None
        key_len = k.size(2)

        # SDPA applies its default 1/sqrt(d_head) scaling. Rescale q first
        # so the user-specified attention_scale is preserved exactly.
        default_scale = 1.0 / math.sqrt(self.d_head)
        if self.attention_scale != default_scale:
            q = q * (self.attention_scale / default_scale)

        attn_mask = None

        if self.is_causal:
            q_positions = torch.arange(
                position_offset,
                position_offset + T,
                device=x.device,
            )
            k_positions = torch.arange(key_len, device=x.device)
            causal_mask = k_positions[None, :] <= q_positions[:, None]
            attn_mask = causal_mask[None, None, :, :]

        if padding_mask is not None:
            padding_mask = padding_mask.to(device=x.device, dtype=torch.bool)
            if padding_mask.shape != (B, key_len):
                raise ValueError(
                    f"Expected padding_mask {(B, key_len)}, "
                    f"got {tuple(padding_mask.shape)}"
                )
            if not padding_mask.any(dim=-1).all():
                raise ValueError("sequence does not have valid key")

            key_mask = padding_mask[:, None, None, :]
            attn_mask = key_mask if attn_mask is None else (attn_mask & key_mask)

        dropout_p = self.attn_dropout.p if self.training else 0.0

        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=dropout_p,
            is_causal=False,
            enable_gqa=(self.n_heads != self.n_kv_heads),
        )

        out = out.transpose(1, 2).reshape(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)

        if use_cache:
            return out, present
        return out


class CrossAttention(nn.Module):
    """Cross-attention implemented with PyTorch SDPA."""
    def __init__(
        self,
        d_model,
        n_heads,
        n_kv_heads,
        max_seq_len,
        dropout=0.0,
        use_rope=False,
        attention_scale=None,
    ):
        super().__init__()

        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0

        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        self.attention_scale = (
            1.0 / math.sqrt(self.d_head)
            if attention_scale is None
            else float(attention_scale)
        )

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(
        self,
        x,
        encoder_inputs,
        source_padding_mask=None,
        past_key_value=None,
        use_cache=False,
    ):
        B, T, C = x.shape
        B_enc, S, _ = encoder_inputs.shape

        if B != B_enc:
            raise ValueError("Encoder and decoder batch sizes must match.")

        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        if past_key_value is None:
            k = self.k_proj(encoder_inputs).view(
                B, S, self.n_kv_heads, self.d_head
            ).transpose(1, 2)
            v = self.v_proj(encoder_inputs).view(
                B, S, self.n_kv_heads, self.d_head
            ).transpose(1, 2)
        else:
            k, v = past_key_value

        present = (k, v) if use_cache else None

        default_scale = 1.0 / math.sqrt(self.d_head)
        if self.attention_scale != default_scale:
            q = q * (self.attention_scale / default_scale)

        attn_mask = None
        if source_padding_mask is not None:
            source_padding_mask = source_padding_mask.to(
                device=x.device, dtype=torch.bool
            )
            if source_padding_mask.shape != (B, S):
                raise ValueError("source_padding_mask must have shape (B, S).")
            if not source_padding_mask.any(dim=-1).all():
                raise ValueError("sequence does not have valid tokens")
            attn_mask = source_padding_mask[:, None, None, :]

        dropout_p = self.attn_dropout.p if self.training else 0.0

        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=dropout_p,
            is_causal=False,
            enable_gqa=(self.n_heads != self.n_kv_heads),
        )

        out = out.transpose(1, 2).reshape(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)

        if use_cache:
            return out, present
        return out


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 512
    n_layers: int = 12
    n_heads: int = 8
    n_kv_heads: int = 8
    d_ff: int = 4096
    max_seq_len: int = 1024
    dropout: float = 0.1
    pos_emb_type: str = "rope"
    attention_scale: float = None


def sinusoidal_positional_encoding(
    max_len: int,
    d_model: int,
) -> torch.Tensor:
    PE = torch.zeros(
        max_len,
        d_model,
    )

    for pos in range(max_len):
        for i in range(
            d_model // 2
        ):
            PE[
                pos,
                2 * i,
            ] = math.sin(
                pos
                / 10000
                ** (2 * i / d_model)
            )
            PE[
                pos,
                2 * i + 1,
            ] = math.cos(
                pos
                / 10000
                ** (2 * i / d_model)
            )

    return PE


class EncoderFeedForward(nn.Module):
    def __init__(
        self,
        d_model,
        d_ff,
        dropout,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(
                d_model,
                d_ff,
            ),
            nn.GELU(),
            nn.Linear(
                d_ff,
                d_model,
            ),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.ln1 = nn.LayerNorm(
            cfg.d_model
        )

        self.attn = MultiHeadAttention(
            cfg.d_model,
            cfg.n_heads,
            cfg.n_kv_heads,
            cfg.max_seq_len,
            cfg.dropout,
            use_rope=(
                cfg.pos_emb_type == "rope"
            ),
            attention_scale=(
                cfg.attention_scale
            ),
        )

        self.ln2 = nn.LayerNorm(
            cfg.d_model
        )

        self.ffn = EncoderFeedForward(
            cfg.d_model,
            cfg.d_ff,
            cfg.dropout,
        )

    def forward(
        self,
        x,
        padding_mask=None,
    ):
        x = x + self.attn(
            self.ln1(x),
            padding_mask=padding_mask,
        )

        x = x + self.ffn(
            self.ln2(x)
        )

        return x


class EncoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.cfg = cfg

        self.tok_emb = nn.Embedding(
            cfg.vocab_size,
            cfg.d_model,
        )

        if cfg.pos_emb_type == "sinusoidal":
            self.register_buffer(
                "pos_emb",
                sinusoidal_positional_encoding(
                    cfg.max_seq_len,
                    cfg.d_model,
                ),
                persistent=False,
            )

        self.blocks = nn.ModuleList(
            [
                EncoderBlock(cfg)
                for _ in range(
                    cfg.n_layers
                )
            ]
        )

        self.ln_f = nn.LayerNorm(
            cfg.d_model
        )

    def forward(
        self,
        idx,
        padding_mask=None,
    ):
        B, T = idx.shape

        if padding_mask is not None:
            if padding_mask.shape != (
                B,
                T,
            ):
                raise ValueError(
                    "padding_mask must have shape "
                    "(B, T)"
                )

            padding_mask = padding_mask.to(
                device=idx.device,
                dtype=torch.bool,
            )

        x = self.tok_emb(idx)

        if self.cfg.pos_emb_type == "sinusoidal":
            x = x + self.pos_emb[
                :T
            ].unsqueeze(0)

        for block in self.blocks:
            if (
                self.training
                and USE_GRAD_CHECKPOINTING
                and torch.is_grad_enabled()
            ):
                x = torch.utils.checkpoint.checkpoint(
                    block,
                    x,
                    padding_mask,
                    use_reentrant=False,
                )
            else:
                x = block(
                    x,
                    padding_mask=padding_mask,
                )

        return self.ln_f(x)


class DecoderFeedForward(nn.Module):
    def __init__(
        self,
        d_model,
        d_ff,
        dropout,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(
                d_model,
                d_ff,
            ),
            nn.GELU(),
            nn.Linear(
                d_ff,
                d_model,
            ),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.ln1 = nn.LayerNorm(
            cfg.d_model
        )

        self.self_attn = MultiHeadAttention(
            cfg.d_model,
            cfg.n_heads,
            cfg.n_kv_heads,
            cfg.max_seq_len,
            cfg.dropout,
            use_rope=(
                cfg.pos_emb_type == "rope"
            ),
            attention_scale=(
                cfg.attention_scale
            ),
            is_causal=True,
        )

        self.ln2 = nn.LayerNorm(
            cfg.d_model
        )

        self.cross_attn = CrossAttention(
            cfg.d_model,
            cfg.n_heads,
            cfg.n_kv_heads,
            cfg.max_seq_len,
            cfg.dropout,
            use_rope=(
                cfg.pos_emb_type == "rope"
            ),
            attention_scale=(
                cfg.attention_scale
            ),
        )

        self.ln3 = nn.LayerNorm(
            cfg.d_model
        )

        self.ffn = DecoderFeedForward(
            cfg.d_model,
            cfg.d_ff,
            cfg.dropout,
        )

    def forward(
        self,
        x,
        encoder_inputs,
        target_padding_mask=None,
        source_padding_mask=None,
        past_key_value=None,
        use_cache=False,
        position_offset=0,
    ):
        if past_key_value is None:
            past_self = None
            past_cross = None
        else:
            past_self, past_cross = (
                past_key_value
            )

        if use_cache:
            attn, present_self = (
                self.self_attn(
                    self.ln1(x),
                    padding_mask=(
                        target_padding_mask
                    ),
                    past_key_value=past_self,
                    use_cache=True,
                    position_offset=(
                        position_offset
                    ),
                )
            )
        else:
            attn = self.self_attn(
                self.ln1(x),
                padding_mask=(
                    target_padding_mask
                ),
            )
            present_self = None

        x = x + attn

        if use_cache:
            cross, present_cross = (
                self.cross_attn(
                    self.ln2(x),
                    encoder_inputs,
                    source_padding_mask=(
                        source_padding_mask
                    ),
                    past_key_value=past_cross,
                    use_cache=True,
                )
            )
        else:
            cross = self.cross_attn(
                self.ln2(x),
                encoder_inputs,
                source_padding_mask=(
                    source_padding_mask
                ),
            )
            present_cross = None

        x = x + cross

        x = x + self.ffn(
            self.ln3(x)
        )

        if use_cache:
            return x, (
                present_self,
                present_cross,
            )

        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.cfg = cfg

        self.tok_emb = nn.Embedding(
            cfg.vocab_size,
            cfg.d_model,
        )

        if cfg.pos_emb_type == "sinusoidal":
            self.register_buffer(
                "pos_emb",
                sinusoidal_positional_encoding(
                    cfg.max_seq_len,
                    cfg.d_model,
                ),
                persistent=False,
            )

        self.blocks = nn.ModuleList(
            [
                DecoderBlock(cfg)
                for _ in range(
                    cfg.n_layers
                )
            ]
        )

        self.ln_f = nn.LayerNorm(
            cfg.d_model
        )

        self.head = nn.Linear(
            cfg.d_model,
            cfg.vocab_size,
            bias=False,
        )

        self.head.weight = (
            self.tok_emb.weight
        )

    def forward(
        self,
        idx,
        encoder_inputs,
        target_padding_mask=None,
        source_padding_mask=None,
        past_key_values=None,
        use_cache=False,
    ):
        B, T = idx.shape

        if (
            past_key_values is not None
            and len(past_key_values)
            != len(self.blocks)
        ):
            raise ValueError(
                "Cache must contain one entry "
                "per decoder layer."
            )

        past_len = (
            0
            if past_key_values is None
            else past_key_values[0][0][0].size(2)
        )

        total_len = past_len + T

        if total_len > self.cfg.max_seq_len:
            raise ValueError(
                f"Decoder length {total_len} exceeds "
                f"max_seq_len={self.cfg.max_seq_len}."
            )

        if target_padding_mask is not None:
            if target_padding_mask.shape != (
                B,
                total_len,
            ):
                raise ValueError(
                    f"Expected target mask {(B, total_len)}, "
                    f"got {tuple(target_padding_mask.shape)}"
                )

            target_padding_mask = (
                target_padding_mask.to(
                    device=idx.device,
                    dtype=torch.bool,
                )
            )

        x = self.tok_emb(idx)

        if self.cfg.pos_emb_type == "sinusoidal":
            end_pos = past_len + T
            x = x + self.pos_emb[
                past_len:end_pos
            ].unsqueeze(0)

        presents = []

        for i, block in enumerate(
            self.blocks
        ):
            past = (
                None
                if past_key_values is None
                else past_key_values[i]
            )

            if use_cache:
                x, present = block(
                    x,
                    encoder_inputs,
                    target_padding_mask=(
                        target_padding_mask
                    ),
                    source_padding_mask=(
                        source_padding_mask
                    ),
                    past_key_value=past,
                    use_cache=True,
                    position_offset=past_len,
                )
                presents.append(present)
            else:
                x = block(
                    x,
                    encoder_inputs,
                    target_padding_mask=(
                        target_padding_mask
                    ),
                    source_padding_mask=(
                        source_padding_mask
                    ),
                )

        logits = self.head(
            self.ln_f(x)
        )

        if use_cache:
            return logits, tuple(
                presents
            )

        return logits

    # PCR helper: return decoder hidden states.
    #
    # MEMORY-SAFE PCR DESIGN:
    # The decoder receives only one BOS token. It therefore:
    #   - performs 1-token causal self-attention
    #   - cross-attends to the full encoder representation
    #   - produces one document representation
    #
    # This avoids the O(T^2) decoder self-attention memory that occurs
    # if the entire 1024-token document is passed to the decoder.
    def encode_representation(
        self,
        idx,
        encoder_inputs,
        source_padding_mask=None,
    ):
        B = idx.size(0)

        decoder_input = torch.full(
            (B, 1),
            BOS_ID,
            dtype=idx.dtype,
            device=idx.device,
        )

        target_padding_mask = torch.ones(
            (B, 1),
            dtype=torch.bool,
            device=idx.device,
        )

        x = self.tok_emb(
            decoder_input
        )

        if self.cfg.pos_emb_type == "sinusoidal":
            x = x + self.pos_emb[
                :1
            ].unsqueeze(0)

        for block in self.blocks:
            if (
                self.training
                and USE_GRAD_CHECKPOINTING
                and torch.is_grad_enabled()
            ):
                x = torch.utils.checkpoint.checkpoint(
                    block,
                    x,
                    encoder_inputs,
                    target_padding_mask,
                    source_padding_mask,
                    None,
                    False,
                    0,
                    use_reentrant=False,
                )
            else:
                x = block(
                    x,
                    encoder_inputs,
                    target_padding_mask=target_padding_mask,
                    source_padding_mask=source_padding_mask,
                )

        return self.ln_f(x)


class EncoderDecoderTransformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.cfg = cfg
        self.encoder = EncoderLM(cfg)
        self.decoder = DecoderLM(cfg)

    def forward(
        self,
        src,
        tgt,
        source_padding_mask=None,
        target_padding_mask=None,
        past_key_values=None,
        use_cache=False,
    ):
        encoder_outputs = self.encoder(
            src,
            padding_mask=source_padding_mask,
        )

        result = self.decoder(
            tgt,
            encoder_outputs,
            target_padding_mask=target_padding_mask,
            source_padding_mask=source_padding_mask,
            past_key_values=past_key_values,
            use_cache=use_cache,
        )

        if use_cache:
            logits, present_key_values = result
            return logits, present_key_values

        return result

    # PCR representation:
    # source document -> encoder -> decoder cross-attention
    # using the same document as decoder input -> final decoder hidden.
    def encode_pcr(
        self,
        token_ids,
    ):
        # Encoder sees the complete 1024-token chunk.
        source_padding_mask = (
            token_ids != PAD_ID
        )

        encoder_outputs = self.encoder(
            token_ids,
            padding_mask=source_padding_mask,
        )

        # Decoder uses only BOS and cross-attends to the encoder.
        # This is the key OOM fix.
        hidden = self.decoder.encode_representation(
            token_ids,
            encoder_outputs,
            source_padding_mask=source_padding_mask,
        )

        return hidden.squeeze(1)


In [13]:
# ============================================================
# STANDARD ADDITIVE LoRA
# ============================================================

class LoRALinear(nn.Module):
    """
    Standard additive LoRA:

        y = W0 x + (alpha / r) B(A(dropout(x)))

    The original linear layer is frozen.
    """
    def __init__(
        self,
        layer,
        rank=16,
        alpha=32,
        dropout=0.05,
    ):
        super().__init__()

        if not isinstance(
            layer,
            nn.Linear,
        ):
            raise TypeError(
                "LoRALinear expects nn.Linear"
            )

        self.layer = layer
        self.rank = int(rank)
        self.alpha = float(alpha)
        self.scaling = (
            self.alpha / self.rank
        )
        self.dropout = nn.Dropout(
            dropout
        )

        for p in self.layer.parameters():
            p.requires_grad_(False)

        self.A = nn.Parameter(
            torch.empty(
                self.rank,
                layer.in_features,
            )
        )

        self.B = nn.Parameter(
            torch.zeros(
                layer.out_features,
                self.rank,
            )
        )

        nn.init.kaiming_uniform_(
            self.A,
            a=math.sqrt(5),
        )

    def forward(self, x):
        base = self.layer(x)

        update = F.linear(
            self.dropout(x),
            self.B @ self.A,
        )

        return base + self.scaling * update


# ============================================================
# Build encoder-decoder model and load YOUR checkpoint
# ============================================================
BASE_CHECKPOINT_PATH = "/kaggle/input/datasets/ksaiveerendraprasad/encoder-decoder-model/encoder_decoder_final.pt"
cfg = LMConfig(
    vocab_size=VOCAB_SIZE,
    d_model=512,
    n_layers=12,
    n_heads=8,
    n_kv_heads=8,
    d_ff=4096,
    max_seq_len=MAX_SEQ_LEN,
    dropout=0.1,
    pos_emb_type="rope",
)

model = EncoderDecoderTransformer(cfg)

print(
    "BASE_CHECKPOINT_PATH:",
    BASE_CHECKPOINT_PATH,
)

if not os.path.exists(
    BASE_CHECKPOINT_PATH
):
    raise FileNotFoundError(
        "\nBase encoder-decoder checkpoint not found:\n"
        f"{BASE_CHECKPOINT_PATH}\n\n"
        "Replace BASE_CHECKPOINT_PATH in the "
        "Configuration cell with your .pt checkpoint."
    )

checkpoint = torch.load(
    BASE_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

if (
    isinstance(checkpoint, dict)
    and "model_state_dict" in checkpoint
):
    state_dict = checkpoint[
        "model_state_dict"
    ]
else:
    state_dict = checkpoint

state_dict = {
    k.replace(
        "module.",
        "",
        1,
    )
    if k.startswith("module.")
    else k: v
    for k, v in state_dict.items()
}

missing, unexpected = model.load_state_dict(
    state_dict,
    strict=False,
)

print(
    "Missing checkpoint keys:",
    len(missing),
)
print(
    "Unexpected checkpoint keys:",
    len(unexpected),
)

# ------------------------------------------------------------
# Inject standard LoRA into:
#
#   Encoder self-attention:
#       q_proj, k_proj, v_proj, out_proj
#
#   Decoder self-attention:
#       q_proj, k_proj, v_proj, out_proj
#
#   Decoder cross-attention:
#       q_proj, k_proj, v_proj, out_proj
# ------------------------------------------------------------

target_names = (
    "q_proj",
    "k_proj",
    "v_proj",
    "out_proj",
)

for block in model.encoder.blocks:
    for name in target_names:
        old_layer = getattr(
            block.attn,
            name,
        )

        setattr(
            block.attn,
            name,
            LoRALinear(
                old_layer,
                rank=LORA_RANK,
                alpha=LORA_ALPHA,
                dropout=LORA_DROPOUT,
            ),
        )

for block in model.decoder.blocks:
    for name in target_names:
        old_self_layer = getattr(
            block.self_attn,
            name,
        )

        setattr(
            block.self_attn,
            name,
            LoRALinear(
                old_self_layer,
                rank=LORA_RANK,
                alpha=LORA_ALPHA,
                dropout=LORA_DROPOUT,
            ),
        )

        old_cross_layer = getattr(
            block.cross_attn,
            name,
        )

        setattr(
            block.cross_attn,
            name,
            LoRALinear(
                old_cross_layer,
                rank=LORA_RANK,
                alpha=LORA_ALPHA,
                dropout=LORA_DROPOUT,
            ),
        )

# Freeze everything except LoRA A/B.
for name, p in model.named_parameters():
    if (
        name.endswith(".A")
        or name.endswith(".B")
    ):
        p.requires_grad_(True)
    else:
        p.requires_grad_(False)

model = model.to(device)

trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"Trainable: {trainable:,}"
)
print(
    f"Total:     {total:,}"
)
print(
    f"Trainable %: "
    f"{100 * trainable / total:.4f}%"
)

BASE_CHECKPOINT_PATH: /kaggle/input/datasets/ksaiveerendraprasad/encoder-decoder-model/encoder_decoder_final.pt
Missing checkpoint keys: 180
Unexpected checkpoint keys: 0
Trainable: 2,359,296
Total:     149,137,408
Trainable %: 1.5820%


In [14]:
# ============================================================
# LoRA optimizer
# ============================================================

lora_a_parameters = []
lora_b_parameters = []

for name, p in model.named_parameters():
    if not p.requires_grad:
        continue

    if name.endswith(".A"):
        lora_a_parameters.append(p)
    elif name.endswith(".B"):
        lora_b_parameters.append(p)

optimizer = torch.optim.AdamW(
    [
        {
            "params": lora_a_parameters,
            "lr": LORA_A_LR,
        },
        {
            "params": lora_b_parameters,
            "lr": LORA_B_LR,
        },
    ],
    weight_decay=WEIGHT_DECAY,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(
        USE_AMP
        and torch.cuda.is_available()
    ),
)

print(
    "LoRA A parameters:",
    sum(
        p.numel()
        for p in lora_a_parameters
    ),
)

print(
    "LoRA B parameters:",
    sum(
        p.numel()
        for p in lora_b_parameters
    ),
)

LoRA A parameters: 1179648
LoRA B parameters: 1179648


In [15]:
# ============================================================
# PCR document encoding
# ============================================================
# For PCR, there is no generation target.
#
# We therefore use each document chunk as:
#   1. encoder input
#   2. decoder input
#
# The decoder receives the encoder representation through
# CROSS-ATTENTION. The final decoder hidden states are pooled
# into the PCR embedding.
#
# This ensures both encoder LoRA and decoder/cross-attention
# LoRA participate in the PCR contrastive objective.
# ============================================================

def encode_query_tensor(
    model,
    query_chunks,
):
    """
    Memory-safe encoder-decoder PCR representation.

    For each document chunk:
      Encoder: full 1024-token chunk.
      Decoder: ONE BOS token that cross-attends to encoder outputs.

    CHUNK_BATCH_SIZE remains 1.
    """
    chunks = query_chunks.to(
        device,
        non_blocking=True,
    )

    valid = (
        chunks != PAD_ID
    ).any(dim=1)

    valid_chunks = chunks[
        valid
    ]

    if valid_chunks.size(0) == 0:
        return torch.zeros(
            model.cfg.d_model,
            device=device,
        )

    chunk_embs = []

    for j in range(
        0,
        valid_chunks.size(0),
        CHUNK_BATCH_SIZE,
    ):
        token_ids = valid_chunks[
            j:j + CHUNK_BATCH_SIZE
        ]

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(
                USE_AMP
                and torch.cuda.is_available()
            ),
        ):
            hidden = model.encode_pcr(
                token_ids
            )

        # hidden is [B, D] because decoder produces one BOS state.
        chunk_embs.append(
            hidden.float()
        )

        del token_ids, hidden

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    chunk_embs = torch.cat(
        chunk_embs,
        dim=0,
    )

    document_embedding = (
        chunk_embs.mean(dim=0)
    )

    return F.normalize(
        document_embedding.float(),
        p=2,
        dim=-1,
    )


@torch.no_grad()
def encode_documents(
    model,
    records,
    document_ids,
    device,
    chunk_batch_size=1,
):
    was_training = model.training
    model.eval()

    document_embeddings = []

    for doc_start in tqdm(
        range(len(document_ids)),
        desc="Encoding documents",
    ):
        doc_id = document_ids[
            doc_start
        ]

        chunks = records[
            doc_id
        ]["chunks"]

        emb = encode_query_tensor(
            model,
            chunks,
        )

        document_embeddings.append(
            emb.cpu()
        )

        del chunks, emb

        if (
            torch.cuda.is_available()
            and doc_start % 10 == 0
        ):
            torch.cuda.empty_cache()

    if was_training:
        model.train()

    return torch.stack(
        document_embeddings,
        dim=0,
    )


In [16]:
# -------------------------
# Multi-positive contrastive loss
# -------------------------
def multi_positive_contrastive_loss(
    similarity_scores,
    positive_mask,
    candidate_mask,
    temperature=0.07,
):
    logits = similarity_scores / temperature

    logits = logits.masked_fill(
        ~candidate_mask,
        float("-inf"),
    )

    positive_logits = logits.masked_fill(
        ~positive_mask,
        float("-inf"),
    )

    valid_rows = (
        positive_mask.any(dim=1)
        & candidate_mask.any(dim=1)
    )

    if not valid_rows.all():
        raise ValueError(
            "Every query must have at least one positive "
            "and one candidate."
        )

    numerator = torch.logsumexp(
        positive_logits,
        dim=1,
    )

    denominator = torch.logsumexp(
        logits,
        dim=1,
    )

    return -(numerator - denominator).mean()


In [17]:
# ============================================================
# Memory-safe PCR training
# ============================================================

def train_one_epoch(
    model,
    train_loader,
    optimizer,
    scaler,
    device,
):
    model.train()

    running_loss = 0.0
    count = 0

    for batch in tqdm(
        train_loader,
        desc="Training",
        leave=False,
    ):
        samples = (
            batch
            if isinstance(batch, list)
            else [batch]
        )

        actual_batch_size = len(
            samples
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        batch_loss = 0.0

        for sample in samples:
            query_chunks = sample[
                "query_chunks"
            ].to(
                device,
                non_blocking=True,
            )

            candidate_chunks = sample[
                "candidate_chunks"
            ].to(
                device,
                non_blocking=True,
            )

            positive_mask = sample[
                "positive_mask"
            ].to(
                device,
                non_blocking=True,
            )

            query_embedding = (
                encode_query_tensor(
                    model,
                    query_chunks,
                )
            )

            candidate_embeddings = []

            for c in range(
                candidate_chunks.size(0)
            ):
                candidate_embedding = (
                    encode_query_tensor(
                        model,
                        candidate_chunks[c],
                    )
                )

                candidate_embeddings.append(
                    candidate_embedding
                )

                del candidate_embedding

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            candidate_embeddings = torch.stack(
                candidate_embeddings,
                dim=0,
            )

            similarities = (
                query_embedding.unsqueeze(0)
                @ candidate_embeddings.transpose(
                    0,
                    1,
                )
            )

            loss = multi_positive_contrastive_loss(
                similarities,
                positive_mask.unsqueeze(0),
                torch.ones_like(
                    positive_mask,
                    dtype=torch.bool,
                ).unsqueeze(0),
                temperature=TEMPERATURE,
            )

            loss_for_backward = (
                loss / actual_batch_size
            )

            if (
                USE_AMP
                and torch.cuda.is_available()
            ):
                scaler.scale(
                    loss_for_backward
                ).backward()
            else:
                loss_for_backward.backward()

            batch_loss += float(
                loss.detach().item()
            )

            del (
                query_chunks,
                candidate_chunks,
                positive_mask,
                query_embedding,
                candidate_embeddings,
                similarities,
                loss,
                loss_for_backward,
            )

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        if (
            USE_AMP
            and torch.cuda.is_available()
        ):
            scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            [
                p
                for p in model.parameters()
                if p.requires_grad
            ],
            GRAD_CLIP,
        )

        if (
            USE_AMP
            and torch.cuda.is_available()
        ):
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()

        running_loss += (
            batch_loss / actual_batch_size
        )

        count += 1

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return running_loss / max(
        count,
        1,
    )

In [18]:
# -------------------------
# Full-pool retrieval evaluation
# -------------------------
@torch.no_grad()
def evaluate_retrieval_full_pool(
    model,
    query_records,
    candidate_records,
    query_data,
    k=7,
    eval_batch_size=1,
):
    model.eval()

    candidate_ids = list(
        candidate_records.keys()
    )

    query_ids = [
        qid
        for qid in query_data.keys()
        if query_data[qid].get(
            "relevant_candidates"
        )
    ]

    candidate_embeddings = (
        encode_documents(
            model,
            candidate_records,
            candidate_ids,
            device,
            chunk_batch_size=1,
        )
    )

    query_embeddings = []

    for qid in tqdm(
        query_ids,
        desc="Encoding queries",
    ):
        emb = encode_query_tensor(
            model,
            query_records[qid]["chunks"],
        )
        query_embeddings.append(
            emb.cpu()
        )

    query_embeddings = torch.stack(
        query_embeddings
    )

    hits = 0
    total_relevant = 0
    total_retrieved = (
        len(query_ids) * k
    )

    predictions = {}

    for i, qid in enumerate(
        query_ids
    ):
        scores = (
            query_embeddings[
                i:i + 1
            ]
            @ candidate_embeddings.T
        ).squeeze(0)

        topk = min(
            k,
            scores.numel(),
        )

        values, indices = torch.topk(
            scores,
            k=topk,
        )

        retrieved = [
            candidate_ids[int(j)]
            for j in indices
        ]

        relevant = set(
            query_data[qid][
                "relevant_candidates"
            ]
        )

        intersection = len(
            set(retrieved)
            & relevant
        )

        hits += intersection
        total_relevant += len(
            relevant
        )

        predictions[qid] = {
            "retrieved_candidates": retrieved,
            "relevant_candidates": list(
                relevant
            ),
            "scores": [
                float(v)
                for v in values
            ],
        }

    precision = (
        hits / total_retrieved
        if total_retrieved
        else 0.0
    )

    recall = (
        hits / total_relevant
        if total_relevant
        else 0.0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    return {
        "Micro-Precision@7": precision,
        "Micro-Recall@7": recall,
        "Micro-F1@7": f1,
    }, predictions


In [19]:
# ============================================================
# Checkpoint helper
# ============================================================

def save_checkpoint(
    path,
    epoch,
    model,
    optimizer,
    scaler,
    train_loss,
    metrics=None,
):
    payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "train_loss": train_loss,
        "metrics": metrics,
        "config": {
            "max_seq_len": MAX_SEQ_LEN,
            "chunk_overlap": CHUNK_OVERLAP,
            "max_chunks": MAX_CHUNKS,
            "num_negatives": NUM_NEGATIVES,
            "chunk_batch_size": CHUNK_BATCH_SIZE,
            "batch_size": BATCH_SIZE,
            "temperature": TEMPERATURE,
            "lora_rank": LORA_RANK,
            "lora_alpha": LORA_ALPHA,
            "lora_dropout": LORA_DROPOUT,
            "lora_a_lr": LORA_A_LR,
            "lora_b_lr": LORA_B_LR,
            "adaptation": (
                "standard additive LoRA + random negatives"
            ),
            "backbone": (
                "encoder-decoder MHA"
            ),
            "pos_emb_type": "rope",
            "decoder_pcr_tokens": 1,
            "oom_fix": "encoder full chunk + 1-token BOS decoder cross-attention",
            "encoder_layers": 12,
            "decoder_layers": 12,
            "d_model": 512,
            "n_heads": 8,
            "n_kv_heads": 8,
            "d_ff": 4096,
        },
    }

    torch.save(
        payload,
        path,
    )

    return path

In [20]:
# -------------------------
# One training epoch
# Memory-efficient LoRA PCR training
# -------------------------
def train_one_epoch(
    model,
    train_loader,
    optimizer,
    scaler,
    device,
):
    model.train()

    running_loss = 0.0
    count = 0

    for batch in tqdm(
        train_loader,
        desc="Training",
        leave=False,
    ):
        # collate_fn returns a list for BATCH_SIZE > 1.
        samples = (
            batch
            if isinstance(batch, list)
            else [batch]
        )

        batch_size_actual = len(
            samples
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        batch_loss = 0.0

        # -------------------------------------------------
        # IMPORTANT:
        # Each query is fully processed and BACKWARDED
        # before moving to the next query.
        #
        # Therefore BATCH_SIZE=2 does NOT keep two complete
        # Transformer activation graphs in memory.
        # -------------------------------------------------
        for sample_idx, sample in enumerate(
            samples
        ):
            query_chunks = sample[
                "query_chunks"
            ].to(
                device,
                non_blocking=True,
            )

            candidate_chunks = sample[
                "candidate_chunks"
            ].to(
                device,
                non_blocking=True,
            )

            positive_mask = sample[
                "positive_mask"
            ].to(
                device,
                non_blocking=True,
            )

            # ---------------------------------------------
            # Query encoding
            # ---------------------------------------------
            query_embedding = (
                encode_query_tensor(
                    model,
                    query_chunks,
                )
            )

            # ---------------------------------------------
            # Candidate encoding
            # ---------------------------------------------
            candidate_embeddings = []

            for c in range(
                candidate_chunks.size(0)
            ):
                candidate_embedding = (
                    encode_query_tensor(
                        model,
                        candidate_chunks[c],
                    )
                )

                candidate_embeddings.append(
                    candidate_embedding
                )

            candidate_embeddings = torch.stack(
                candidate_embeddings,
                dim=0,
            )

            similarities = (
                query_embedding.unsqueeze(0)
                @ candidate_embeddings.transpose(
                    0,
                    1,
                )
            )

            loss = (
                multi_positive_contrastive_loss(
                    similarities,
                    positive_mask.unsqueeze(0),
                    torch.ones_like(
                        positive_mask,
                        dtype=torch.bool,
                    ).unsqueeze(0),
                    temperature=TEMPERATURE,
                )
            )

            # Average the gradients over queries in this batch.
            loss_for_backward = (
                loss / batch_size_actual
            )

            if USE_AMP and torch.cuda.is_available():
                scaler.scale(
                    loss_for_backward
                ).backward()
            else:
                loss_for_backward.backward()

            batch_loss += float(
                loss.detach().item()
            )

            # Free everything belonging to this query.
            del (
                query_chunks,
                candidate_chunks,
                positive_mask,
                query_embedding,
                candidate_embeddings,
                similarities,
                loss,
                loss_for_backward,
            )

            # The autograd graph from the query is released after backward.
            # This keeps the peak memory approximately independent of BATCH_SIZE.

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # ---------------------------------------------
        # One optimizer update for the whole batch
        # ---------------------------------------------
        if USE_AMP and torch.cuda.is_available():
            scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            [
                p
                for p in model.parameters()
                if p.requires_grad
            ],
            GRAD_CLIP,
        )

        if USE_AMP and torch.cuda.is_available():
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()

        batch_loss /= batch_size_actual

        running_loss += batch_loss
        count += 1

        del samples

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return (
        running_loss
        / max(count, 1)
    )


In [21]:
# -------------------------
# Full-pool retrieval evaluation
# -------------------------
@torch.no_grad()
def evaluate_retrieval_full_pool(
    model,
    query_records,
    candidate_records,
    query_data,
    k=7,
    eval_batch_size=1,
):
    model.eval()

    candidate_ids = list(
        candidate_records.keys()
    )

    query_ids = [
        qid
        for qid in query_data.keys()
        if query_data[qid].get(
            "relevant_candidates"
        )
    ]

    candidate_embeddings = (
        encode_documents(
            model,
            candidate_records,
            candidate_ids,
            device,
            chunk_batch_size=1,
        )
    )

    query_embeddings = []

    for qid in tqdm(
        query_ids,
        desc="Encoding queries",
    ):
        emb = encode_query_tensor(
            model,
            query_records[qid]["chunks"],
        )
        query_embeddings.append(
            emb.cpu()
        )

    query_embeddings = torch.stack(
        query_embeddings
    )

    hits = 0
    total_relevant = 0
    total_retrieved = (
        len(query_ids) * k
    )

    predictions = {}

    for i, qid in enumerate(
        query_ids
    ):
        scores = (
            query_embeddings[
                i:i + 1
            ]
            @ candidate_embeddings.T
        ).squeeze(0)

        topk = min(
            k,
            scores.numel(),
        )

        values, indices = torch.topk(
            scores,
            k=topk,
        )

        retrieved = [
            candidate_ids[int(j)]
            for j in indices
        ]

        relevant = set(
            query_data[qid][
                "relevant_candidates"
            ]
        )

        intersection = len(
            set(retrieved)
            & relevant
        )

        hits += intersection
        total_relevant += len(
            relevant
        )

        predictions[qid] = {
            "retrieved_candidates": retrieved,
            "relevant_candidates": list(
                relevant
            ),
            "scores": [
                float(v)
                for v in values
            ],
        }

    precision = (
        hits / total_retrieved
        if total_retrieved
        else 0.0
    )

    recall = (
        hits / total_relevant
        if total_relevant
        else 0.0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    return {
        "Micro-Precision@7": precision,
        "Micro-Recall@7": recall,
        "Micro-F1@7": f1,
    }, predictions


In [22]:
# -------------------------
# Checkpoint helper
# -------------------------
def save_checkpoint(
    path,
    epoch,
    model,
    optimizer,
    scaler,
    train_loss,
    metrics=None,
):
    payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "train_loss": train_loss,
        "metrics": metrics,
        "config": {
            "max_seq_len": MAX_SEQ_LEN,
            "chunk_overlap": CHUNK_OVERLAP,
            "max_chunks": MAX_CHUNKS,
            "num_negatives": NUM_NEGATIVES,
            "chunk_batch_size": CHUNK_BATCH_SIZE,
            "temperature": TEMPERATURE,
            "lora_rank": LORA_RANK,
            "lora_alpha": LORA_ALPHA,
            "lora_dropout": LORA_DROPOUT,
            "lora_a_lr": LORA_A_LR,
            "lora_b_lr": LORA_B_LR,
            "adaptation": "standard additive LoRA + random negatives",
            "backbone": "encoder-decoder MHA",
        },
    }

    torch.save(
        payload,
        path,
    )

    return path


In [23]:
BATCH_SIZE

8

In [24]:
# -------------------------
# Train
# -------------------------
best_f1 = -1.0
history = []

for epoch in range(
    1,
    EPOCHS + 1,
):
    print(
        f"\n===== Epoch {epoch}/{EPOCHS} ====="
    )

    clear_cuda_memory()
    gpu_memory_report(
        "Before train | "
    )

    train_loss = train_one_epoch(
        model=model,
        train_loader=train_loader,
        optimizer=optimizer,
        scaler=scaler,
        device=device,
    )

    gpu_memory_report(
        "After train | "
    )

    epoch_checkpoint = os.path.join(
        CHECKPOINT_DIR,
        f"epoch_{epoch}.pt",
    )

    save_checkpoint(
        epoch_checkpoint,
        epoch,
        model,
        optimizer,
        scaler,
        train_loss,
        metrics=None,
    )

    dev_metrics, dev_predictions = (
        evaluate_retrieval_full_pool(
            model=model,
            query_records=dev_query_records,
            candidate_records=dev_candidate_records,
            query_data=dev_data,
            k=7,
            eval_batch_size=1,
        )
    )

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        **dev_metrics,
    }

    history.append(row)

    with open(
        HISTORY_PATH,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            history,
            f,
            indent=2,
        )

    print(
        f"Train loss: {train_loss:.6f}"
    )
    print(
        f"Precision@7: "
        f"{dev_metrics['Micro-Precision@7']:.6f}"
    )
    print(
        f"Recall@7: "
        f"{dev_metrics['Micro-Recall@7']:.6f}"
    )
    print(
        f"F1@7: "
        f"{dev_metrics['Micro-F1@7']:.6f}"
    )

    current_f1 = dev_metrics[
        "Micro-F1@7"
    ]

    if current_f1 > best_f1:
        best_f1 = current_f1

        save_checkpoint(
            BEST_CHECKPOINT,
            epoch,
            model,
            optimizer,
            scaler,
            train_loss,
            metrics=dev_metrics,
        )

        with open(
            os.path.join(
                CHECKPOINT_DIR,
                "best_dev_predictions.json",
            ),
            "w",
            encoding="utf-8",
        ) as f:
            json.dump(
                dev_predictions,
                f,
                indent=2,
            )

        print(
            "Saved new best checkpoint:",
            BEST_CHECKPOINT,
        )

    clear_cuda_memory()



===== Epoch 1/5 =====
Before train | GPU 0: allocated=0.57 GB, reserved=0.57 GB, free=13.89 GB / total=14.56 GB
Before train | GPU 1: allocated=0.00 GB, reserved=0.00 GB, free=14.46 GB / total=14.56 GB


Training:   0%|          | 0/103 [00:00<?, ?it/s]

After train | GPU 0: allocated=0.61 GB, reserved=0.64 GB, free=13.78 GB / total=14.56 GB
After train | GPU 1: allocated=0.00 GB, reserved=0.00 GB, free=14.46 GB / total=14.56 GB


Encoding documents:   0%|          | 0/1023 [00:00<?, ?it/s]

Encoding queries:   0%|          | 0/118 [00:00<?, ?it/s]

Train loss: 0.691032
Precision@7: 0.069007
Recall@7: 0.058582
F1@7: 0.063369
Saved new best checkpoint: /kaggle/working/PCR_EncoderDecoder_LoRA_Random_Negatives/best.pt

===== Epoch 2/5 =====
Before train | GPU 0: allocated=0.61 GB, reserved=0.64 GB, free=13.78 GB / total=14.56 GB
Before train | GPU 1: allocated=0.00 GB, reserved=0.00 GB, free=14.46 GB / total=14.56 GB


... (16 outputs left out)

After train | GPU 0: allocated=0.61 GB, reserved=0.64 GB, free=13.78 GB / total=14.56 GB
After train | GPU 1: allocated=0.00 GB, reserved=0.00 GB, free=14.46 GB / total=14.56 GB


Encoding documents:   0%|          | 0/1023 [00:00<?, ?it/s]

Encoding queries:   0%|          | 0/118 [00:00<?, ?it/s]

Train loss: 0.331371
Precision@7: 0.124697
Recall@7: 0.105858
F1@7: 0.114508
Saved new best checkpoint: /kaggle/working/PCR_EncoderDecoder_LoRA_Random_Negatives/best.pt


In [25]:
# -------------------------
# Final history
# -------------------------
print(json.dumps(history, indent=2))
print("Best Dev F1@7:", best_f1)
print("Best checkpoint:", BEST_CHECKPOINT)


[
  {
    "epoch": 1,
    "train_loss": 0.6910316787877129,
    "Micro-Precision@7": 0.06900726392251816,
    "Micro-Recall@7": 0.05858170606372045,
    "Micro-F1@7": 0.06336853807670928
  },
  {
    "epoch": 2,
    "train_loss": 0.5345595004489121,
    "Micro-Precision@7": 0.09564164648910412,
    "Micro-Recall@7": 0.08119218910585817,
    "Micro-F1@7": 0.08782657031684268
  },
  {
    "epoch": 3,
    "train_loss": 0.3816518378489226,
    "Micro-Precision@7": 0.1089588377723971,
    "Micro-Recall@7": 0.09249743062692703,
    "Micro-F1@7": 0.1000555864369094
  },
  {
    "epoch": 4,
    "train_loss": 0.34671377094046585,
    "Micro-Precision@7": 0.1198547215496368,
    "Micro-Recall@7": 0.10174717368961973,
    "Micro-F1@7": 0.11006114508060032
  },
  {
    "epoch": 5,
    "train_loss": 0.3313706852857349,
    "Micro-Precision@7": 0.12469733656174334,
    "Micro-Recall@7": 0.10585817060637205,
    "Micro-F1@7": 0.11450806003335186
  }
]
Best Dev F1@7: 0.11450806003335186
Best checkpoin

In [26]:


def load_best_lora_checkpoint(
    model,
    checkpoint_path,
):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    state_dict = checkpoint[
        "model_state_dict"
    ]

    missing, unexpected = model.load_state_dict(
        state_dict,
        strict=True,
    )

    print(
        "Loaded best checkpoint:",
        checkpoint_path,
    )
    print(
        "Checkpoint epoch:",
        checkpoint.get("epoch"),
    )
    print(
        "Best dev metrics:",
        checkpoint.get("metrics"),
    )

    if missing or unexpected:
        raise RuntimeError(
            f"Checkpoint mismatch. "
            f"Missing={missing}, "
            f"Unexpected={unexpected}"
        )

    return checkpoint


@torch.no_grad()
def generate_test_retrieval(
    model,
    query_records,
    candidate_records,
    test_data,
    top_k=7,
):
    model.eval()

    candidate_ids = list(
        candidate_records.keys()
    )

    # --------------------------------------------------------
    # Encode ALL test candidates.
    # This is done one document at a time to avoid OOM.
    # --------------------------------------------------------
    candidate_embeddings = encode_documents(
        model,
        candidate_records,
        candidate_ids,
        device,
        chunk_batch_size=1,
    )

    # --------------------------------------------------------
    # Encode test queries.
    # --------------------------------------------------------
    query_ids = list(
        test_data.keys()
    )

    predictions = {}

    for qid in tqdm(
        query_ids,
        desc="Generating test retrieval",
    ):
        query_embedding = encode_query_tensor(
            model,
            query_records[qid]["chunks"],
        ).cpu()

        scores = (
            query_embedding.unsqueeze(0)
            @ candidate_embeddings.T
        ).squeeze(0)

        k = min(
            top_k,
            scores.numel(),
        )

        values, indices = torch.topk(
            scores,
            k=k,
        )

        retrieved_candidates = [
            candidate_ids[int(idx)]
            for idx in indices
        ]

        predictions[str(qid)] = {
            "retrieved_candidates":
                retrieved_candidates,
            "scores": [
                float(v)
                for v in values
            ],
        }

        del query_embedding, scores

    return predictions


# Clear memory before loading the best checkpoint.
clear_cuda_memory()

best_checkpoint = load_best_lora_checkpoint(
    model,
    BEST_CHECKPOINT,
)

model = model.to(device)
model.eval()

test_predictions = generate_test_retrieval(
    model=model,
    query_records=test_query_records,
    candidate_records=test_candidate_records,
    test_data=test_data,
    top_k=7,
)

with open(
    TEST_PREDICTIONS_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        test_predictions,
        f,
        indent=2,
    )

print(
    f"\nSaved {len(test_predictions)} test query predictions."
)
print(
    "Output:",
    TEST_PREDICTIONS_PATH,
)

# Example
first_qid = next(
    iter(test_predictions)
)
print(
    "\nExample test prediction:"
)
print(
    first_qid,
    test_predictions[first_qid],
)


Loaded best checkpoint: /kaggle/working/PCR_EncoderDecoder_LoRA_Random_Negatives/best.pt
Checkpoint epoch: 5
Best dev metrics: {'Micro-Precision@7': 0.12469733656174334, 'Micro-Recall@7': 0.10585817060637205, 'Micro-F1@7': 0.11450806003335186}


Encoding documents:   0%|          | 0/1727 [00:00<?, ?it/s]

Generating test retrieval:   0%|          | 0/237 [00:00<?, ?it/s]


Saved 237 test query predictions.
Output: /kaggle/working/PCR_EncoderDecoder_LoRA_Random_Negatives/test_retrieval_predictions.json

Example test prediction:
0000021652 {'retrieved_candidates': ['0000021652', '0001696091', '0001460234', '0000312114', '0001221178', '0000590455', '0001102670'], 'scores': [1.0000001192092896, 0.9978829026222229, 0.9968637228012085, 0.9964097738265991, 0.9961829781532288, 0.9959840178489685, 0.9957398176193237]}


In [27]:
# ============================================================
# FINAL TEST RETRIEVAL GENERATION
# ============================================================
# Loads the best PCR LoRA checkpoint and retrieves Top-7
# candidates from the COMPLETE test candidate pool.
#
# Output:
# /kaggle/working/PCR_EncoderDecoder_LoRA_Random_Negatives/
# test_retrieval_predictions.json
# ============================================================

def load_best_lora_checkpoint(
    model,
    checkpoint_path,
):
    if not os.path.exists(
        checkpoint_path
    ):
        raise FileNotFoundError(
            f"Best PCR checkpoint not found: "
            f"{checkpoint_path}\n"
            "Run training first or set BEST_CHECKPOINT."
        )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    state_dict = checkpoint[
        "model_state_dict"
    ]

    missing, unexpected = (
        model.load_state_dict(
            state_dict,
            strict=True,
        )
    )

    if missing or unexpected:
        raise RuntimeError(
            f"Checkpoint mismatch.\n"
            f"Missing: {missing}\n"
            f"Unexpected: {unexpected}"
        )

    print(
        "Loaded best checkpoint:",
        checkpoint_path,
    )
    print(
        "Epoch:",
        checkpoint.get("epoch"),
    )
    print(
        "Dev metrics:",
        checkpoint.get("metrics"),
    )

    return checkpoint


@torch.no_grad()
def generate_test_retrieval(
    model,
    query_records,
    candidate_records,
    test_data,
    top_k=7,
):
    model.eval()

    candidate_ids = list(
        candidate_records.keys()
    )

    candidate_embeddings = (
        encode_documents(
            model,
            candidate_records,
            candidate_ids,
            device,
            chunk_batch_size=1,
        )
    )

    predictions = {}

    for qid in tqdm(
        list(test_data.keys()),
        desc="Generating test retrieval",
    ):
        query_embedding = (
            encode_query_tensor(
                model,
                query_records[qid]["chunks"],
            )
            .cpu()
        )

        scores = (
            query_embedding.unsqueeze(0)
            @ candidate_embeddings.T
        ).squeeze(0)

        k = min(
            top_k,
            scores.numel(),
        )

        values, indices = torch.topk(
            scores,
            k=k,
        )

        retrieved = [
            candidate_ids[int(i)]
            for i in indices
        ]

        predictions[str(qid)] = {
            "retrieved_candidates": retrieved,
            "scores": [
                float(v)
                for v in values
            ],
        }

        del (
            query_embedding,
            scores,
        )

    return predictions


clear_cuda_memory()

best_checkpoint = load_best_lora_checkpoint(
    model,
    BEST_CHECKPOINT,
)

model = model.to(device)
model.eval()

test_predictions = (
    generate_test_retrieval(
        model=model,
        query_records=test_query_records,
        candidate_records=test_candidate_records,
        test_data=test_data,
        top_k=7,
    )
)

with open(
    TEST_PREDICTIONS_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        test_predictions,
        f,
        indent=2,
    )

print(
    f"Saved {len(test_predictions)} "
    "test query predictions."
)
print(
    "Output:",
    TEST_PREDICTIONS_PATH,
)

Loaded best checkpoint: /kaggle/working/PCR_EncoderDecoder_LoRA_Random_Negatives/best.pt
Epoch: 5
Dev metrics: {'Micro-Precision@7': 0.12469733656174334, 'Micro-Recall@7': 0.10585817060637205, 'Micro-F1@7': 0.11450806003335186}


Encoding documents:   0%|          | 0/1727 [00:00<?, ?it/s]

Generating test retrieval:   0%|          | 0/237 [00:00<?, ?it/s]

Saved 237 test query predictions.
Output: /kaggle/working/PCR_EncoderDecoder_LoRA_Random_Negatives/test_retrieval_predictions.json
